## **TMDB Movie Data Analysis using Spark and APIs**


In [1]:
import os
import time
import json
import sys
import requests
from dotenv import load_dotenv
from pyspark.sql import SparkSession, functions as F
import matplotlib.pyplot as plt
from pathlib import Path


### *Functions from scripts*

In [2]:
sys.path.append(os.path.abspath("..")) 
from src.tmdb_client import fetch_movie_with_credits
from src.bronze_to_spark import read_bronze_json
import src.silver_transform as st

Let us start the Spark session.

In [3]:
spark = SparkSession.builder.appName("tmdb_lab").getOrCreate()
spark

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/02/03 08:01:12 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


## 1. **Fetching the movie data from the API**

In this project, we will follow the medallion workflow to ingest, transform and analyze the movie data for the wanted ids.

In [4]:
load_dotenv()

TMDB_API_KEY = os.getenv("TMDB_API_KEY")

print(f"The TMDB API Key is fetched")

The TMDB API Key is fetched


In [5]:
# TMDB base URLs and Endpoints
BASE_URL = "https://api.themoviedb.org/3"
MOVIE_ENDPOINT = f"{BASE_URL}/movie"
CREDITS_ENDPOINT = f"{BASE_URL}/movie"

# Movie IDs to fetch
movie_ids = [
    0, 299534, 19995, 140607, 299536, 597, 135397, 420818,
    24428, 168259, 99861, 284054, 12445, 181808, 330457,
    351286, 109445, 321612, 260513
]

len(movie_ids), movie_ids[:5]


(19, [0, 299534, 19995, 140607, 299536])

### **Bronze Layer:**

#### **Bronze Ingestion**
Let us ingest our raw data as JSON first

In [6]:
PROJECT_ROOT = Path(os.getcwd()).parent
out_dir = PROJECT_ROOT / "data" / "bronze" / "tmdb_raw_json"
out_dir.mkdir(parents=True, exist_ok=True)

movie_ids = [
    0, 299534, 19995, 140607, 299536, 597, 135397, 420818,
    24428, 168259, 99861, 284054, 12445, 181808, 330457,
    351286, 109445, 321612, 260513
]


In [7]:

for mid in movie_ids:
    out_path = out_dir / f"movie_id={mid}.json"

    if out_path.exists():
        print(f"Exists, skipping: {mid}")
        continue

    print(f"Fetching: {mid}")
    bundle = fetch_movie_with_credits(mid, TMDB_API_KEY)

    if bundle is None:
        print(f"Movie with Id: {mid} not found")
        continue

    with open(out_path, "w", encoding="utf-8") as f:
        json.dump(bundle, f, ensure_ascii=False)

Fetching: 0
[HTTP 404] Failed: https://api.themoviedb.org/3/movie/0 | {'success': False, 'status_code': 6, 'status_message': 'Invalid id: The pre-requisite id is invalid or not found.'}
ID 0: Movie not found. Skipping
Movie with Id: 0 not found
Exists, skipping: 299534
Exists, skipping: 19995
Exists, skipping: 140607
Exists, skipping: 299536
Exists, skipping: 597
Exists, skipping: 135397
Exists, skipping: 420818
Exists, skipping: 24428
Exists, skipping: 168259
Exists, skipping: 99861
Exists, skipping: 284054
Exists, skipping: 12445
Exists, skipping: 181808
Exists, skipping: 330457
Exists, skipping: 351286
Exists, skipping: 109445
Exists, skipping: 321612
Exists, skipping: 260513


In [8]:
len(list(out_dir.glob("*.json"))) # Let us see the number of moies fetched

18

#### **Bronze to Spark**
Now we can read the Bronze JSON into Spark

In [9]:
bronze_path = str(out_dir)
bronze_df = read_bronze_json(spark, bronze_path)

In [10]:
bronze_df.printSchema()

bronze_df.select("movie.id", "movie.title", "credits.id","fetched_at").show(5, truncate=False)

print("Bronze count:", bronze_df.count())


root
 |-- credits: struct (nullable = true)
 |    |-- cast: array (nullable = true)
 |    |    |-- element: struct (containsNull = true)
 |    |    |    |-- adult: boolean (nullable = true)
 |    |    |    |-- cast_id: long (nullable = true)
 |    |    |    |-- character: string (nullable = true)
 |    |    |    |-- credit_id: string (nullable = true)
 |    |    |    |-- gender: long (nullable = true)
 |    |    |    |-- id: long (nullable = true)
 |    |    |    |-- known_for_department: string (nullable = true)
 |    |    |    |-- name: string (nullable = true)
 |    |    |    |-- order: long (nullable = true)
 |    |    |    |-- original_name: string (nullable = true)
 |    |    |    |-- popularity: double (nullable = true)
 |    |    |    |-- profile_path: string (nullable = true)
 |    |-- crew: array (nullable = true)
 |    |    |-- element: struct (containsNull = true)
 |    |    |    |-- adult: boolean (nullable = true)
 |    |    |    |-- credit_id: string (nullable = true)
 |

26/02/03 08:01:33 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


Bronze count: 18


All our movies are now stored into a spark data frame

## 2. **Silver Layer: Data Cleaning and Preprocessing**

Since we have our data in a dataframe now we can go ahead and clean it according to what we need from it
### **Data Preparation and Cleaning**

#### **Droping irrelevant columns**
Rather than removing unnecessary fields after loading the data, we applied column projection at read time by explicitly selecting only the attributes required for analysis. Columns such as `adult`, `imdb_id`, `original_title`, `video`, and `homepage` were excluded by not selecting them from the nested TMDB JSON structure.

In [11]:
# Keeping only the columns we need (Droping everything else)
base = st.build_silver(bronze_df)


#### **Evaluating JSON-like columns**
TMDB returns several attributes as nested JSON objects (structs) or lists of objects (arrays of structs) rather than flat scalar values. We inspected their Spark schema using `printSchema()` to understand their internal structure before transforming them into analysis-friendly columns.

In [12]:
base.select(
    "belongs_to_collection_raw",
    "genres_raw",
    "spoken_languages_raw",
    "production_countries_raw",
    "production_companies_raw"
).printSchema()

root
 |-- belongs_to_collection_raw: struct (nullable = true)
 |    |-- backdrop_path: string (nullable = true)
 |    |-- id: long (nullable = true)
 |    |-- name: string (nullable = true)
 |    |-- poster_path: string (nullable = true)
 |-- genres_raw: array (nullable = true)
 |    |-- element: struct (containsNull = true)
 |    |    |-- id: long (nullable = true)
 |    |    |-- name: string (nullable = true)
 |-- spoken_languages_raw: array (nullable = true)
 |    |-- element: struct (containsNull = true)
 |    |    |-- english_name: string (nullable = true)
 |    |    |-- iso_639_1: string (nullable = true)
 |    |    |-- name: string (nullable = true)
 |-- production_countries_raw: array (nullable = true)
 |    |-- element: struct (containsNull = true)
 |    |    |-- iso_3166_1: string (nullable = true)
 |    |    |-- name: string (nullable = true)
 |-- production_companies_raw: array (nullable = true)
 |    |-- element: struct (containsNull = true)
 |    |    |-- id: long (nullab

From the schema:

- `belongs_to_collection_raw` is a struct containing fields such as `id` and `name` → we extract `name`.

- `genres_raw`, `production_countries_raw`, and `production_companies_raw` are arrays of structs where the main descriptive field is `name`.

- `spoken_languages_raw` is an array of structs that includes both `name` and `english_name` → we typically use `english_name` for consistent readability.

This schema inspection step ensures we extract the correct fields and avoid null/incorrect transformations when flattening the data into Silver format.